In [7]:
import langchain

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

### Simple LLM Call With Streaming

In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

In [11]:
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026F487ADBE0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026F48C29130>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [12]:
messages = [
    SystemMessage("You are a helpful AI assistant."),
    HumanMessage("What are the top 2 benefits of using Langchain?")
]

In [13]:
## invoke the model with the messages
response = model.invoke(messages)
response

AIMessage(content='**Top\u202f2 Benefits of Using LangChain**\n\n| Benefit | Why It Matters |\n|---------|----------------|\n| **Rapid Development of LLM‑driven applications** | LangChain provides a modular, plug‑and‑play framework for chaining together prompts, models, memory, tools, and data sources. You can prototype complex workflows (e.g., retrieval‑augmented generation, multi‑step reasoning, tool‑calling agents) with just a few lines of code, dramatically cutting development time compared to building everything from scratch. |\n| **Built‑in Integration & Extensibility** | The library ships with adapters for the major LLM providers (OpenAI, Anthropic, Cohere, Llama\u202f2, etc.), vector stores (FAISS, Pinecone, Weaviate, Chroma, etc.), and external tools (APIs, databases, browsers). Its extensible component architecture lets you swap out or extend any piece (prompt templates, retrievers, memory, callbacks) without rewriting the whole pipeline, making it easy to scale, customize, a

In [14]:
print(response.content)

**Top 2 Benefits of Using LangChain**

| Benefit | Why It Matters |
|---------|----------------|
| **Rapid Development of LLM‑driven applications** | LangChain provides a modular, plug‑and‑play framework for chaining together prompts, models, memory, tools, and data sources. You can prototype complex workflows (e.g., retrieval‑augmented generation, multi‑step reasoning, tool‑calling agents) with just a few lines of code, dramatically cutting development time compared to building everything from scratch. |
| **Built‑in Integration & Extensibility** | The library ships with adapters for the major LLM providers (OpenAI, Anthropic, Cohere, Llama 2, etc.), vector stores (FAISS, Pinecone, Weaviate, Chroma, etc.), and external tools (APIs, databases, browsers). Its extensible component architecture lets you swap out or extend any piece (prompt templates, retrievers, memory, callbacks) without rewriting the whole pipeline, making it easy to scale, customize, and keep up with new models or serv

In [15]:
## stream the response
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

**Top 2 Benefits of Using LangChain**

| Benefit | Why It Matters | How It Helps You |
|---------|----------------|------------------|
| **Modular, Plug‑and‑Play Architecture** | LangChain breaks the LLM workflow into reusable components (prompt templates, memory, agents, retrievers, etc.). You can swap out a model, vector store, or tool without rewriting the whole pipeline. | • Faster prototyping – drop in a new LLM or database in a single line of code.<br>• Cleaner codebases – each piece has a clear interface, making maintenance and collaboration easier. |
| **Built‑in Integration with External Tools & Data Sources** | It provides ready‑made connectors to popular vector stores (Pinecone, FAISS, Chroma), APIs (OpenAI, Anthropic, Cohere), and frameworks (LLM‑powered agents, tool‑calling, retrieval‑augmented generation). | • Turn raw data (documents, APIs, spreadsheets) into searchable knowledge for your LLM instantly.<br>• Enable sophisticated use‑cases like chat‑bots that can browse t

### Dynamic Prompt Templates

In [16]:
from langchain_core.prompts import ChatPromptTemplate

 ## create a translation chatbot

translation_template = ChatPromptTemplate.from_messages([
    ("system","You are a professional translator.Translate the following text \"{text}\" from {input_language} to {output_language}."),
    ("user","{text}")
])

prompt = translation_template.invoke({
    "text": "Hello, how are you?",
    "input_language": "English",
    "output_language": "French"
})

In [17]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator.Translate the following text "Hello, how are you?" from English to French.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})])

In [18]:
translated_response=model.invoke(prompt)
print(translated_response.content)

Bonjour, comment ça va ?


### Building Chain

In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
def create_story_chain():
    ## Template for story generation
    story_template = ChatPromptTemplate.from_messages([
        ("system","You are a professional and creative storyteller. Write a short and engaging story based on the given theme, character, and setting."),
        ("user","Theme: {theme}\n Main Character: {main_character}\n Setting: {setting}")
    ])

    ## Template for story analysis
    analysis_template = ChatPromptTemplate.from_messages([
        ("system","You are an expert story analyst. Analyze the following story and provide feedback."),
        ("user","{story}")
    ])

    story_chain=(
        story_template | model | StrOutputParser()
    )

    def analyze_story(story_text):
        return {"story" : story_text}

    analysis_chain=(
        story_chain | RunnableLambda(analyze_story) | analysis_template | model | StrOutputParser()
    )

    return analysis_chain

In [20]:
chain = create_story_chain()
chain

ChatPromptTemplate(input_variables=['main_character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a professional and creative storyteller. Write a short and engaging story based on the given theme, character, and setting.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['main_character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\n Main Character: {main_character}\n Setting: {setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, '

In [21]:
result=chain.invoke({
    "theme": "Friendship and Adventure",
    "main_character": "Alex",
    "setting": "A magical forest"
})

print(result)

**Story Analysis – “The Whispering Pines”**

---

## 1. Overview  

*The Whispering Pines* is a bright, myth‑like adventure that follows Alex, an ordinary human who stumbles into an enchanted forest (Elderwood) and is tasked with reviving its dying Heart Tree. The narrative is structured around three classic “trial” episodes—courage, trust, imagination—each embodied by a vivid set‑piece and a symbolic key. The tale blends lyrical description with a clear moral arc, ending with a satisfying resolution that ties the external quest to an internal growth in friendship and self‑understanding.

---

## 2. Strengths  

| Aspect | What Works Well | Why It Matters |
|--------|----------------|----------------|
| **World‑building** | The forest feels alive: sapphire‑blue moss, “crystalline shadows,” singing mushrooms, a rope‑bridge of luminous vines, etc. | Sensory richness immerses the reader and makes the magical setting feel concrete rather than generic. |
| **Imagery & Language** | Use of co